In [1]:
import torch
import torch.nn as nn
import math
import numpy as np


In [14]:
# фиксируем seed чтобы результаты повторялись
torch.manual_seed(0)

T = 10   # количество состояний энкодера
N = 4   # размерность вектора

ES = torch.randn(T, N)
DS = torch.randn(N)

print("Encoder states:")
print(ES.shape)
print(ES)

print("\nDecoder state:")
print(DS.shape)
print(DS)

Encoder states:
torch.Size([10, 4])
tensor([[-1.1258, -1.1524, -0.2506, -0.4339],
        [ 0.8487,  0.6920, -0.3160, -2.1152],
        [ 0.3223, -1.2633,  0.3500,  0.3081],
        [ 0.1198,  1.2377,  1.1168, -0.2473],
        [-1.3527, -1.6959,  0.5667,  0.7935],
        [ 0.5988, -1.5551, -0.3414,  1.8530],
        [-0.2159, -0.7425,  0.5627,  0.2596],
        [-0.1740, -0.6787,  0.9383,  0.4889],
        [ 1.2032,  0.0845, -1.2001, -0.0048],
        [-0.5181, -0.3067, -1.5810,  1.7066]])

Decoder state:
torch.Size([4])
tensor([-0.0209, -0.7185,  0.5186, -1.3125])


### задание 1
Напишите класс, который будет вычислять сходство вектора контекста декодера и векторов контекста энкодера. Сходство в этом задании определим как sim(h,s)

In [27]:
class Similarity1(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, encoder_states: torch.Tensor, decoder_state: torch.Tensor):
        # encoder_states: [T, N]
        # decoder_state:  [N]

        sim = torch.matmul(encoder_states, decoder_state)  # [T]

        return sim

In [28]:
sim = Similarity1()
res = sim(ES,DS)
res

tensor([ 1.2910,  2.0975,  0.6780,  0.0120,  0.4991, -1.5044,  0.4891,  0.3363,
        -0.7020, -2.8288])

### задание 2
Напишите класс, который будет вычислять сходство вектора контекста декодера и векторов контекста энкодера. Сходство в этом задании определим как sim(h,s)

In [83]:
class Similarity2(nn.Module):
    def __init__(self, encoder_dim: int, decoder_dim: int, intermediate_dim: int):
        super().__init__()

        self.fc1 = nn.Linear(encoder_dim, intermediate_dim)
        self.fc2 = nn.Linear(decoder_dim, intermediate_dim)
        self.fc3 = nn.Linear(intermediate_dim, 1)
        self.tanh = nn.Tanh()


    def forward(self, encoder_state: torch.Tensor, decoder_state: torch.Tensor):
        sim = self.fc3(self.tanh(self.fc1(encoder_state) + self.fc2(decoder_state).unsqueeze(0)))
        return sim

In [86]:
sim = Similarity2(ES.shape[1],DS.shape[0],32)
res = sim(ES,DS).T
res

tensor([[ 0.0992,  0.0284,  0.0867, -0.2205, -0.0105,  0.1428,  0.0256, -0.0062,
          0.2822,  0.0610]], grad_fn=<PermuteBackward0>)